# TinyChirp CNN build pipeline\n
\n
Trains one CNN on log-mel spectrograms, exports an int8 TFLite model, and writes a Rust audio sample file.

In [4]:
from typing import TYPE_CHECKING
from utils import (
    TARGET_FRAMES_MEL,
    get_paths,
    configure_tf_runtime,
    set_global_seed,
)

if TYPE_CHECKING:
    import keras
else:
    from tensorflow import keras

configure_tf_runtime()
set_global_seed()

MODEL_STEM = "cnn_mel_tf"
paths = get_paths(MODEL_STEM)
OUT_TFLITE = paths.out_tflite
OUT_AUDIO_RS = paths.out_audio_rs
BATCH_SIZE = 32



In [5]:
from utils import (
    make_mel_datasets,
    NUM_MEL_BINS_MEL,
)

train_ds, val_ds, test_ds, label_names = make_mel_datasets(
    num_mel_bins=NUM_MEL_BINS_MEL,
    target_frames=TARGET_FRAMES_MEL,
)
num_labels = len(label_names)
print("Classes:", label_names)



Found 11292 files belonging to 2 classes.
Found 1380 files belonging to 2 classes.
Found 1393 files belonging to 2 classes.
Classes: ['non_target' 'target']


In [6]:
from utils import NUM_MEL_BINS_MEL, TARGET_FRAMES_MEL, init_wandb, get_callbacks, finish_wandb

CONV_FILTER_SIZE = 3
N_CHANNELS = 4

TARGET_FRAMES = TARGET_FRAMES_MEL
NUM_MEL_BINS = NUM_MEL_BINS_MEL

end_of_conv1_s1 = (TARGET_FRAMES - CONV_FILTER_SIZE + 1) // 2
end_of_conv2_s1 = (end_of_conv1_s1 - CONV_FILTER_SIZE + 1) // 2
end_of_conv1_s2 = (NUM_MEL_BINS - CONV_FILTER_SIZE + 1) // 2
end_of_conv2_s2 = (end_of_conv1_s2 - CONV_FILTER_SIZE + 1) // 2

model = keras.Sequential([
    keras.layers.Input(shape=(TARGET_FRAMES, NUM_MEL_BINS, 1)),
    keras.layers.Conv2D(N_CHANNELS, (CONV_FILTER_SIZE, CONV_FILTER_SIZE), activation="relu"),
    keras.layers.AveragePooling2D((2, 2)),
    keras.layers.Conv2D(N_CHANNELS, (CONV_FILTER_SIZE, CONV_FILTER_SIZE), activation="relu"),
    keras.layers.AveragePooling2D((2, 2)),
    keras.layers.Reshape((end_of_conv2_s2 * end_of_conv2_s1 * N_CHANNELS,)),
    keras.layers.Dense(8, activation="relu"),
    keras.layers.Dense(num_labels),
])
model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 182, 78, 4)     │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_2             │ (None, 91, 39, 4)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 89, 37, 4)      │           148 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_3             │ (None, 44, 18, 4)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_1 (Reshape)             │ (None, 3168)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │        25,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │            18 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,558 (99.84 KB)

 Trainable params: 25,558 (99.84 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:

init_wandb(MODEL_STEM, config={
    "conv_filter_size": CONV_FILTER_SIZE,
    "n_channels": N_CHANNELS,
})

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=get_callbacks(10,5,BATCH_SIZE)
)
finish_wandb()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/nathan/.netrc.
wandb: Currently logged in as: nathan-duboisset (nathan-duboisset-cole-polytechnique) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1/100


I0000 00:00:1776677134.298254   59281 service.cc:145] XLA service 0x79b8d0003910 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776677134.298294   59281 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9


 21/353 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.5765 - loss: 0.9142

I0000 00:00:1776677136.441916   59281 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


353/353 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.8483 - loss: 0.3585

In [ ]:
from utils import (
    build_representative_batches,
    export_keras_model_to_int8_tflite,
)

val_specs = build_representative_batches(test_ds, take=100)

try:
    export_keras_model_to_int8_tflite(model, val_specs, OUT_TFLITE)
    print(f"Success! Wrote {OUT_TFLITE}")
except Exception as e:
    print(f"Conversion failed: {e}")

In [ ]:
from utils import evaluate_tflite_model

train_m, test_m, avg_ms = evaluate_tflite_model(OUT_TFLITE, MODEL_STEM, train_ds, test_ds)
print(f"Avg inference: {avg_ms:.3f} ms")